<a href="https://colab.research.google.com/github/kartik69-tech/Fabric-Grading-App/blob/main/ml_model_train_model_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model, save_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import joblib

# Synthetic data generation
def generate_synthetic_data(samples=2000):
    np.random.seed(42)

    # Synthetic features
    defect_count = np.random.randint(0, 11, samples)  # 0-10 defects
    color_variance = np.round(np.random.uniform(0, 1, samples), 2)  # 0-1.0
    texture_variance = np.round(np.random.uniform(0, 1, samples), 2)  # 0-1.0

    # Assign grades based on criteria (1-4)
    grade_defects = np.where(defect_count == 0, 1,
                            np.where(defect_count <= 2, 2,
                                    np.where(defect_count <= 5, 3, 4)))

    grade_color = np.where(color_variance < 0.1, 1,
                          np.where(color_variance < 0.3, 2,
                                  np.where(color_variance < 0.6, 3, 4)))

    grade_texture = np.where(texture_variance < 0.1, 1,
                            np.where(texture_variance < 0.3, 2,
                                    np.where(texture_variance < 0.6, 3, 4)))

    data = pd.DataFrame({
        'defects': defect_count,
        'color_var': color_variance,
        'texture_var': texture_variance,
        'grade_defects': grade_defects,
        'grade_color': grade_color,
        'grade_texture': grade_texture
    })
    return data

# Build and train model
def train_and_save_model():
    # Generate synthetic data
    data = generate_synthetic_data(samples=2000)

    # Features and labels
    X = data[['defects', 'color_var', 'texture_var']]
    y = data[['grade_defects', 'grade_color', 'grade_texture']]

    # Scale features
    scaler = MinMaxScaler()
    X_scaled = scaler.fit_transform(X)

    # Convert grades to categorical (one-hot encoding)
    y_defects = to_categorical(y['grade_defects'] - 1, num_classes=4)
    y_color = to_categorical(y['grade_color'] - 1, num_classes=4)
    y_texture = to_categorical(y['grade_texture'] - 1, num_classes=4)

    # Split data into training and testing sets
    X_train, X_test, y_train_defects, y_test_defects, y_train_color, y_test_color, y_train_texture, y_test_texture = train_test_split(
        X_scaled, y_defects, y_color, y_texture, test_size=0.2, random_state=42
    )

    # Build the model
    input_layer = Input(shape=(3,))
    dense = Dense(64, activation='relu')(input_layer)
    dense = Dropout(0.2)(dense)

    # Separate output layers for each attribute
    output_defects = Dense(4, activation='softmax', name='defects')(dense)
    output_color = Dense(4, activation='softmax', name='color')(dense)
    output_texture = Dense(4, activation='softmax', name='texture')(dense)

    model = Model(
        inputs=input_layer,
        outputs=[output_defects, output_color, output_texture]
    )

    # Compile the model
    model.compile(
        optimizer='adam',
        loss={'defects': 'categorical_crossentropy',
              'color': 'categorical_crossentropy',
              'texture': 'categorical_crossentropy'},
        metrics={'defects': 'accuracy', 'color': 'accuracy', 'texture': 'accuracy'}
    )

    # Train the model
    history = model.fit(
        X_train,
        {'defects': y_train_defects, 'color': y_train_color, 'texture': y_train_texture},
        epochs=20,
        batch_size=32,
        validation_split=0.2
    )

    # Save the model and scaler
    save_model(model, 'ml_model/fabric_grader.h5')
    joblib.dump(scaler, 'ml_model/scaler.pkl')

    print("Model and scaler saved successfully!")

if __name__ == '__main__':
    train_and_save_model()

Epoch 1/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - color_accuracy: 0.1821 - color_loss: 1.4265 - defects_accuracy: 0.3414 - defects_loss: 1.3371 - loss: 4.1194 - texture_accuracy: 0.3623 - texture_loss: 1.3557 - val_color_accuracy: 0.4062 - val_color_loss: 1.3124 - val_defects_accuracy: 0.4062 - val_defects_loss: 1.2525 - val_loss: 3.8151 - val_texture_accuracy: 0.4375 - val_texture_loss: 1.2502
Epoch 2/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - color_accuracy: 0.4609 - color_loss: 1.2846 - defects_accuracy: 0.4354 - defects_loss: 1.2240 - loss: 3.7369 - texture_accuracy: 0.4403 - texture_loss: 1.2284 - val_color_accuracy: 0.4938 - val_color_loss: 1.2096 - val_defects_accuracy: 0.4062 - val_defects_loss: 1.1566 - val_loss: 3.5094 - val_texture_accuracy: 0.4281 - val_texture_loss: 1.1432
Epoch 3/20
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - color_accuracy: 0.5010 - color_loss: 1.1765 - defects_accuracy: 0.4332 - defects_loss: 1.1347 - loss: 3.4571 - texture_accuracy: 0.4315 - texture_l

Model and scaler saved successfully!
